In [1]:
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark context
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
# inspect spark session
spark

In [3]:
import os
import pandas as pd
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import yfinance as yf

## Computing the independent table with Tickers

In [4]:
# Folder with the raw Excel price history files
stock_data_dir = "../../Data/Stock_data"

# Path where you want to store the result (optional)
OUTPUT_PARQUET = "../Data/Cleaned_data/daily_data_tickers.parquet"
# get all files in directory

# Date column name in the Excel files (adapt if needed)
DATE_COL = "Local Date"   # or "Date" if yours is different

In [5]:
files = os.listdir(stock_data_dir)

In [6]:
def get_ticker_from_excel(path):
    # Read only column A
    df = pd.read_excel(path, header=None)

    # Ticker is in cell A4  → row index 3, col index 0
    ticker = df.iloc[3, 0]

    # Clean ticker if needed (strip spaces etc.)
    ticker = str(ticker).strip()

    return ticker

In [7]:
def load_excel_file(path, ticker):
    # Load entire excel file to inspect structure
    df_temp = pd.read_excel(path, sheet_name=0) 
    df_temp.columns = [i for i in range(df_temp.shape[1])]

    # Find 'Local Date' in columns 0 or 2 to determine file structure
    for col in [0, 2]:
        if col < len(df_temp.columns) and (df_temp[col] == 'Local Date').any():
            header_row = (df_temp[col] == 'Local Date').idxmax() + 1
            usecols = 'A:I' if col == 0 else 'C:K'  # Skip exchange columns if present
            break
    else:
        raise ValueError("Could not find 'Local Date' column")
    
    # Read actual data and convert to Spark DataFrame
    df = pd.read_excel(path, sheet_name=0, header=header_row, usecols=usecols)
    # Add ticker column
    df["ticker"] = ticker

    return df

## Exploring

In [8]:
file_1 = os.path.join(stock_data_dir, files[0])

In [9]:
df_1 = load_excel_file(path=file_1, ticker='ARG.TO')
df_1.head()

,Local Date,Local Time,Close,Net,%Chg,Open,Low,High,Volume,ticker
0,2024-03-22 21:11:00,2024-03-22 21:11:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARG.TO
1,2024-03-22 21:03:00,2024-03-22 21:03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARG.TO
2,2024-03-22 21:02:00,2024-03-22 21:02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARG.TO
3,2024-03-22 21:01:00,2024-03-22 21:01:00,1.48,0.0,0.0,1.48,1.48,1.48,400.0,ARG.TO
4,2024-03-22 21:00:00,2024-03-22 21:00:00,1.48,0.0,0.0,1.48,1.48,1.48,1500.0,ARG.TO


In [10]:
df_1.describe()

,Local Date,Local Time,Close,Net,%Chg,Open,Low,High,Volume
count,9729,9729,2461.000000,2460.000000,2460.000000,2461.000000,2461.000000,2461.000000,2461.000000
mean,2024-02-21 05:55:39.284613120,2024-02-21 05:55:39.284613120,1.345656,0.000041,0.000040,1.345512,1.344935,1.346252,3394.229988
min,2024-01-15 07:01:00,2024-01-15 07:01:00,1.220000,-0.045000,-0.033210,1.220000,1.220000,1.220000,100.000000
25%,2024-02-01 19:30:00,2024-02-01 19:30:00,1.290000,0.000000,0.000000,1.290000,1.290000,1.290000,100.000000
50%,2024-02-22 21:59:00,2024-02-22 21:59:00,1.320000,0.000000,0.000000,1.320000,1.320000,1.320000,400.000000
75%,2024-03-12 14:33:00,2024-03-12 14:33:00,1.390000,0.000000,0.000000,1.390000,1.390000,1.390000,3000.000000
max,2024-03-22 21:11:00,2024-03-22 21:11:00,1.530000,0.040000,0.030303,1.525000,1.525000,1.530000,113500.000000
std,NaN,NaN,0.078731,0.006297,0.004756,0.078767,0.078708,0.078783,8081.321384


## FE

In [11]:
def aggregate_to_daily(sdf):
    sdf = sdf.withColumn("Local Date (ET)", F.to_date("Local Date (ET)"))
    
    daily = (
        sdf.groupBy("ticker", "Local Date (ET)")
           .agg(
               F.first("Open").alias("Open"),   # if already daily, this is fine
               F.max("High").alias("High"),
               F.min("Low").alias("Low"),
               F.last("Close").alias("Close"),
               F.sum("Volume").alias("Volume")
           )
    )

    return daily

In [12]:
def compute_daily_features(sdf):
    """
    Given a Spark DataFrame with columns:
        ticker, Local Date (ET), Open, High, Low, Close, Volume
    compute daily-level features needed for biweekly aggregation.
    """

    # Sort window per ticker by date
    w_ticker_date = Window.partitionBy("ticker").orderBy("Local Date (ET)")

    # close-to-close return (for beta)
    sdf = sdf.withColumn(
        "daily_return",
        (F.col("Close") / F.lag("Close").over(w_ticker_date)) - 1
    )
    
    sdf = sdf.withColumn(
        "%Chg",
        F.when(F.lag("Close").over(w_ticker_date).isNotNull(),
               (F.col("Close") / F.lag("Close").over(w_ticker_date) - 1) * 100.0)
    )

    # open-to-close return (momentum)
    sdf = sdf.withColumn(
        "intraday_return",
        (F.col("Close") - F.col("Open")) / F.col("Open")
    )

    # Intraday range (relative)
    sdf = sdf.withColumn(
        "intraday_range",
        (F.col("High") - F.col("Low")) / F.col("Open")
    )

    # Gap (open vs previous close)
    sdf = sdf.withColumn(
        "gap",
        (F.col("Open") / F.lag("Close").over(w_ticker_date)) - 1
    )

    # Up/down day flags
    sdf = sdf.withColumn(
        "up_day",
        F.when(F.col("intraday_return") > 0, 1).otherwise(0)
    ).withColumn(
        "down_day",
        F.when(F.col("intraday_return") < 0, 1).otherwise(0)
    )

    # Up/down volume (just a helper for later)
    sdf = sdf.withColumn(
        "volume_up",
        F.when(F.col("intraday_return") > 0, F.col("Volume")).otherwise(F.lit(None))
    ).withColumn(
        "volume_down",
        F.when(F.col("intraday_return") < 0, F.col("Volume")).otherwise(F.lit(None))
    )

    return sdf

In [13]:
def add_biweekly_periods(sdf):
    """
    Adds:
        - days_since_start
        - biweekly_index (int)
        - biweekly_period (e.g. 'P0', 'P1', ...)
    based on the earliest date across all tickers.
    """
    min_date = sdf.agg(F.min("Local Date (ET)").alias("min_date")).collect()[0]["min_date"]

    sdf = sdf.withColumn(
        "days_since_start",
        F.datediff(F.col("Local Date (ET)"), F.lit(min_date))
    )

    sdf = sdf.withColumn(
        "biweekly_index",
        (F.col("days_since_start") / F.lit(14)).cast("int")
    )

    sdf = sdf.withColumn(
        "biweekly_period",
        F.concat(F.lit("P"), F.col("biweekly_index"))
    )

    return sdf


In [14]:
def aggregate_to_biweekly(sdf):
    """
    Input Spark DF:
        ticker, Local Date (ET), biweekly_period, Open, High, Low, Close, Volume,
        daily_return, intraday_range, gap, up_day, down_day, volume_up, volume_down

    Output biweekly-level DF with metrics like:
        period_start, period_end, period_open, period_close,
        period_high, period_low, avg_close, period_return,
        price_volatility, return_volatility, avg_intraday_range,
        max_intraday_range, total_volume, avg_daily_volume, max_daily_volume,
        volume_volatility, volume_concentration, avg_volume_up_days,
        avg_volume_down_days, gap_count, avg_gap_size, max_gap,
        up_days, down_days

    (event flags you can add later by joining announcements)
    """

    grouped = sdf.groupBy("ticker", "biweekly_index", "biweekly_period", "longname", "industry", "benchmark")

    biweekly = grouped.agg(
        # start / end
        F.min("Local Date (ET)").alias("period_start"),
        F.max("Local Date (ET)").alias("period_end"),

        # open = first open in period (sorted by date)
        F.first("Open", ignorenulls=True).alias("period_open"),
        # close = last close in period
        F.last("Close", ignorenulls=True).alias("period_close"),

        # high/low over the period
        F.max("High").alias("period_high"),
        F.min("Low").alias("period_low"),

        # averages
        F.avg("Close").alias("avg_close"),
        F.avg("%Chg").alias("avg_%Chg")

        # total period return (close / open - 1)
        ((F.last("Close", ignorenulls=True) /
          F.first("Open", ignorenulls=True)) - 1).alias("period_return"),
        
        # total benchmark return (benchmark_close / benchmark_open - 1)
        ((F.last("benchmark_close", ignorenulls=True) /
          F.first("benchmark_open", ignorenulls=True)) - 1).alias("benchmark_period_return"),
        
        # Beta Configuration
         (
            F.covar_samp("daily_return", "benchmark_daily_return") /
            F.var_samp("benchmark_daily_return")
        ).alias("beta"),
        
        # volatilities
        F.stddev_samp("Close").alias("price_volatility"),
        F.stddev_samp("intraday_return").alias("return_volatility"),

        # intraday range stats
        F.avg("intraday_range").alias("avg_intraday_range"),
        F.max("intraday_range").alias("max_intraday_range"),

        # volume stats
        F.sum("Volume").alias("total_volume"),
        F.avg("Volume").alias("avg_daily_volume"),
        F.max("Volume").alias("max_daily_volume"),
        F.stddev_samp("Volume").alias("volume_volatility"),

        # up/down volume
        F.avg("volume_up").alias("avg_volume_up_days"),
        F.avg("volume_down").alias("avg_volume_down_days"),

        # gap stats
        F.sum(F.when(F.col("gap").isNotNull(), 1).otherwise(0)).alias("gap_count"),
        F.avg("gap").alias("avg_gap_size"),
        F.max("gap").alias("max_gap"),

        # up/down day counts
        F.sum("up_day").alias("up_days"),
        F.sum("down_day").alias("down_days"),
    )
    
    biweekly = biweekly.sort("ticker", "biweekly_index")

    return biweekly


In [15]:
all_pandas = []

for file in files:
    if not file.lower().endswith((".xlsx", ".xls")):
        continue
    if file.startswith("~$"):  # temp/lock files
        continue

    path = os.path.join(stock_data_dir, file)

    try:
        ticker = get_ticker_from_excel(path)
    except Exception as e:
        print(f"⚠️ Skipping {file}: error reading ticker from A4 → {e}")
        continue

    print(f"Loading {file} → ticker: {ticker}")

    try:
        df = load_excel_file(path, ticker)
    except Exception as e:
        print(f"⚠️ Skipping {file}: error reading price table → {e}")
        continue

    all_pandas.append(df)

# Combine all pandas DataFrames into one
if not all_pandas:
    raise RuntimeError("No valid Excel files found or all failed to load.")

combined_pdf = pd.concat(all_pandas, ignore_index=True)

combined_pdf.head()

Loading Price History_ARG_20250112_1657_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20250112_1704_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20250112_1706_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20250112_1708_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20250112_1712_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20250223_1252_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20250419_2042_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20250624_0901_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20250831_1223_EIGEN.xlsx → ticker: ARG.TO
Loading Price History_ARG_20251010_1341.xlsx → ticker: ARG.TO
Loading Price History_AYA_20251014_1128_GOLD.xlsx → ticker: AYA.TO
Loading Price History_AYA_20251014_1130_GOLD.xlsx → ticker: AYA.TO
Loading Price History_AYA_20251014_1133_GOLD.xlsx → ticker: AYA.TO
Loading Price History_AYA_20251014_1136_GOLD.xlsx → ticker: AYA.TO
Loading Price History_CGC_20251010_1220 2.xlsx → ticker: C

,Local Date,Local Time,Close,Net,%Chg,Open,Low,High,Volume,ticker
0,2024-03-22 21:11:00,2024-03-22 21:11:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARG.TO
1,2024-03-22 21:03:00,2024-03-22 21:03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARG.TO
2,2024-03-22 21:02:00,2024-03-22 21:02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARG.TO
3,2024-03-22 21:01:00,2024-03-22 21:01:00,1.48,0.0,0.0,1.48,1.48,1.48,400.0,ARG.TO
4,2024-03-22 21:00:00,2024-03-22 21:00:00,1.48,0.0,0.0,1.48,1.48,1.48,1500.0,ARG.TO


In [16]:
# Create Spark DF
sdf = spark.createDataFrame(combined_pdf)

In [17]:
# overwrite the Local Date column with ET time instead of CET 
sdf_with_et = sdf.withColumn("Local Date (ET)",
                                F.from_utc_timestamp(
                                    F.to_utc_timestamp(F.col("Local Date"), "CET"), 
                                    "America/New_York")) \
                    .withColumnRenamed("Local Time", "Local Date (CET)") \
                    .drop("Local Date")

# compare with Local Time column (should be )
sdf_with_et.show()

+-------------------+-----+--------------------+--------------------+-----+-----+-----+------+------+-------------------+
|   Local Date (CET)|Close|                 Net|                %Chg| Open|  Low| High|Volume|ticker|    Local Date (ET)|
+-------------------+-----+--------------------+--------------------+-----+-----+-----+------+------+-------------------+
|2024-03-22 21:11:00|  NaN|                 NaN|                 NaN|  NaN|  NaN|  NaN|   NaN|ARG.TO|2024-03-22 16:11:00|
|2024-03-22 21:03:00|  NaN|                 NaN|                 NaN|  NaN|  NaN|  NaN|   NaN|ARG.TO|2024-03-22 16:03:00|
|2024-03-22 21:02:00|  NaN|                 NaN|                 NaN|  NaN|  NaN|  NaN|   NaN|ARG.TO|2024-03-22 16:02:00|
|2024-03-22 21:01:00| 1.48|                 0.0|                 0.0| 1.48| 1.48| 1.48| 400.0|ARG.TO|2024-03-22 16:01:00|
|2024-03-22 21:00:00| 1.48|                 0.0|                 0.0| 1.48| 1.48| 1.48|1500.0|ARG.TO|2024-03-22 16:00:00|
|2024-03-22 20:59:00| 1.

In [18]:
# Remove rows containing NaN values in volume column
sdf_with_et = sdf_with_et.filter((F.col('Volume').isNotNull()) & (~F.isnan(F.col('Volume')))) \
    .select('Local Date (CET)','Local Date (ET)','ticker', 'Close', 'Net', '%Chg', 'Open', 'High', 'Low', 'Volume')
# Show the cleaned DataFrame
sdf_with_et.show()

+-------------------+-------------------+------+-----+--------------------+--------------------+-----+-----+-----+-------+
|   Local Date (CET)|    Local Date (ET)|ticker|Close|                 Net|                %Chg| Open| High|  Low| Volume|
+-------------------+-------------------+------+-----+--------------------+--------------------+-----+-----+-----+-------+
|2024-03-22 21:01:00|2024-03-22 16:01:00|ARG.TO| 1.48|                 0.0|                 0.0| 1.48| 1.48| 1.48|  400.0|
|2024-03-22 21:00:00|2024-03-22 16:00:00|ARG.TO| 1.48|                 0.0|                 0.0| 1.48| 1.48| 1.48| 1500.0|
|2024-03-22 20:59:00|2024-03-22 15:59:00|ARG.TO| 1.48|                 0.0|                 0.0| 1.48| 1.48| 1.48|  200.0|
|2024-03-22 20:58:00|2024-03-22 15:58:00|ARG.TO| 1.48| 0.00499999999999989|  0.0033898305084745| 1.48| 1.48| 1.48|  600.0|
|2024-03-22 20:57:00|2024-03-22 15:57:00|ARG.TO|1.475|-0.00499999999999989|-0.00337837837837831|1.475|1.475|1.475|  100.0|
|2024-03-22 20:5

In [19]:
# Aggregate to Daily
raw_daily = aggregate_to_daily(sdf_with_et)

# Add Daily features
sdf_daily = compute_daily_features(raw_daily)

# Add biweekly labels
sdf_daily = add_biweekly_periods(sdf_daily)


In [20]:
sdf_daily.show()

+------+---------------+----+-----+----+-----+--------+--------------------+-------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+
|ticker|Local Date (ET)|Open| High| Low|Close|  Volume|        daily_return|               %Chg|     intraday_return|      intraday_range|                 gap|up_day|down_day|volume_up|volume_down|days_since_start|biweekly_index|biweekly_period|
+------+---------------+----+-----+----+-----+--------+--------------------+-------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+
|ARG.TO|     2024-01-15|1.36| 1.38|1.35| 1.38| 71200.0|                NULL|               NULL|0.014705882352941025| 0.02205882352941162|                NULL|     1|       0|  71200.0|       NULL|               0|             0|             P0|
|ARG.TO|     202

In [ ]:
#def aggregate_change_to_biweekly(sdf):
#    grouped = sdf.groupBy("ticker", "biweekly_index", "biweekly_period")
#
#    biweekly = grouped.agg(
#        F.avg("%Chg").alias("avg_%Chg")
#    )
#    
#    biweekly_change = biweekly.sort("ticker", "biweekly_index")
#
#    return biweekly_change

In [ ]:
biweekly_change = aggregate_change_to_biweekly(sdf_daily)
biweekly_change.show()

+------+--------------+---------------+--------------------+
|ticker|biweekly_index|biweekly_period|            avg_%Chg|
+------+--------------+---------------+--------------------+
|ARG.TO|             0|             P0|  -0.305810109744827|
|ARG.TO|             1|             P1| -0.5971767724768162|
|ARG.TO|             2|             P2|  0.2758950896125816|
|ARG.TO|             3|             P3| 0.08927499458281644|
|ARG.TO|             4|             P4|  1.2325078746517593|
|ARG.TO|             5|             P5|  0.6520675714893737|
|ARG.TO|             6|             P6|  1.2934333128453068|
|ARG.TO|             7|             P7|0.025956881642292772|
|ARG.TO|             8|             P8|  0.7381104787800041|
|ARG.TO|             9|             P9| -0.6851516892693033|
|ARG.TO|            10|            P10| -1.1020973375798648|
|ARG.TO|            11|            P11|  -0.093496082472726|
|ARG.TO|            12|            P12|  0.9282145437944267|
|ARG.TO|            13| 

In [22]:
# Save as Parquet
#sdf_daily.write.mode("overwrite").parquet(OUTPUT_PARQUET)

# Optional: also export a pandas version for quick exploration
daily_df = sdf_daily
#daily_pdf.to_csv("../Data/Cleaned_data/daily_data_tickers.csv", index=False)

## Add Industry Category to DAILY data
so we can merge scraped daily benchmark df with initial df

In [63]:
clean_df_dir = "../Data/Cleaned_data/daily_data_tickers.parquet"

In [23]:
#daily_df =  spark.read.parquet(clean_df_dir)
daily_df.show()

+------+---------------+----+-----+----+-----+--------+--------------------+-------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+
|ticker|Local Date (ET)|Open| High| Low|Close|  Volume|        daily_return|               %Chg|     intraday_return|      intraday_range|                 gap|up_day|down_day|volume_up|volume_down|days_since_start|biweekly_index|biweekly_period|
+------+---------------+----+-----+----+-----+--------+--------------------+-------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+
|ARG.TO|     2024-01-15|1.36| 1.38|1.35| 1.38| 71200.0|                NULL|               NULL|0.014705882352941025| 0.02205882352941162|                NULL|     1|       0|  71200.0|       NULL|               0|             0|             P0|
|ARG.TO|     202

In [65]:
df_dates = (
    daily_df.groupBy("ticker")
      .agg(
          F.min("Local Date (ET)").alias("min_period_start"),
          F.max("Local Date (ET)").alias("max_period_end")
      )
)

df_dates.show()


+-------+----------------+--------------+
| ticker|min_period_start|max_period_end|
+-------+----------------+--------------+
| ARG.TO|      2024-01-15|    2025-08-29|
|  ESK.V|      2024-07-18|    2025-08-29|
|  FNM.V|      2024-04-29|    2025-08-29|
|FURY.TO|      2024-07-18|    2025-08-29|
| FOM.TO|      2024-10-15|    2025-10-10|
| FINX.V|      2025-06-18|    2025-10-15|
| ELO.TO|      2024-01-15|    2025-08-29|
|  EDG.V|      2024-07-22|    2025-08-29|
| FVL.TO|      2024-01-18|    2025-08-29|
|  CGC.V|      2024-10-10|    2025-10-09|
|  CKG.V|      2024-10-10|    2025-10-09|
|   DV.V|      2024-09-17|    2025-09-17|
| AYA.TO|      2024-10-15|    2025-10-10|
|  DMX.V|      2024-01-18|    2025-10-15|
| FDY.TO|      2024-10-15|    2025-10-15|
|  FWZ.V|      2024-07-25|    2025-08-29|
|  FAS.V|      2024-10-16|    2025-10-15|
|  CG.TO|      2024-10-10|    2025-10-09|
|    EQX|      2024-07-02|    2025-08-29|
+-------+----------------+--------------+



In [67]:
start = df_dates.select(F.min("min_period_start")).collect()[0][0]
end = df_dates.select(F.max("max_period_end")).collect()[0][0]
print("Start date: ", start, "\nEnd date: ", end)

Start date:  2024-01-15 
End date:  2025-10-15


Use yahoo api to get industry, sector and country

In [24]:
tickers_df = pd.read_csv("../../Data/tickers.csv")

In [25]:
tickers_df

,tickers,Company Name
0,ARG.TO,Amerigo Resources Ltd
1,AYA.TO,Aya Gold & Silver Inc
2,CG.TO,Centerra Gold Inc
3,CGC.V,Canadian Gold Corp
4,CKG.V,Chesapeake Gold Corp
5,DMX.V,District Metals Corp
6,DV.V,Dolly Varden Silver Corp
7,EDG.V,Endurance Gold Corp
8,ELO.TO,Eloro Resources Ltd
9,EQX,Equinox Gold Corp


In [26]:
base_info = []

for ticker in tickers_df["tickers"]:
    t = yf.Ticker(ticker)
    info = t.info
    
    base_info.append({
        "ticker": ticker,
        "longName": info.get("longName"),
        "sector": info.get("sector"),
        "industry": info.get("industry"),
        "country": info.get("country"),
    })

base_info_df = pd.DataFrame(base_info)
base_info_df

,ticker,longName,sector,industry,country
0,ARG.TO,Amerigo Resources Ltd.,Basic Materials,Copper,Canada
1,AYA.TO,Aya Gold & Silver Inc.,Basic Materials,Silver,Canada
2,CG.TO,Centerra Gold Inc.,Basic Materials,Gold,Canada
3,CGC.V,Canadian Gold Corp.,Basic Materials,Gold,Canada
4,CKG.V,Chesapeake Gold Corp.,Basic Materials,Other Precious Metals & Mining,Canada
5,DMX.V,District Metals Corp.,Basic Materials,Other Industrial Metals & Mining,Canada
6,DV.V,Dolly Varden Silver Corporation,Basic Materials,Silver,Canada
7,EDG.V,Endurance Gold Corporation,Basic Materials,Gold,Canada
8,ELO.TO,Eloro Resources Ltd.,Basic Materials,Gold,Canada
9,EQX,Equinox Gold Corp.,Basic Materials,Gold,Canada


In [71]:
base_info_df['country'].value_counts()

country
Canada    19
Name: count, dtype: int64

In [72]:
base_info_df.describe()

,ticker,longName,sector,industry,country
count,19,19,19,19,19
unique,19,19,1,5,1
top,ARG.TO,Amerigo Resources Ltd.,Basic Materials,Gold,Canada
freq,1,1,19,6,19


Only industry is relevant (drop sector and country, no differentiation)

In [27]:
base_info_df = base_info_df[['ticker', 'longName','industry']]
base_info_df.head()

,ticker,longName,industry
0,ARG.TO,Amerigo Resources Ltd.,Copper
1,AYA.TO,Aya Gold & Silver Inc.,Silver
2,CG.TO,Centerra Gold Inc.,Gold
3,CGC.V,Canadian Gold Corp.,Gold
4,CKG.V,Chesapeake Gold Corp.,Other Precious Metals & Mining


In [28]:
spark_base_info = spark.createDataFrame(base_info_df)
spark_base_info.printSchema()

root
 |-- ticker: string (nullable = true)
 |-- longName: string (nullable = true)
 |-- industry: string (nullable = true)



In [29]:
daily_df_enriched = (
    daily_df.join(
        spark_base_info,
        on="ticker",
        how="left"
    )
)
daily_df_enriched.show()

+------+---------------+----+-----+----+-----+--------+--------------------+-------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+--------------------+--------+
|ticker|Local Date (ET)|Open| High| Low|Close|  Volume|        daily_return|               %Chg|     intraday_return|      intraday_range|                 gap|up_day|down_day|volume_up|volume_down|days_since_start|biweekly_index|biweekly_period|            longName|industry|
+------+---------------+----+-----+----+-----+--------+--------------------+-------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+--------------------+--------+
|ARG.TO|     2024-01-15|1.36| 1.38|1.35| 1.38| 71200.0|                NULL|               NULL|0.014705882352941025| 0.02205882352941162|                NULL|     1|      

In [30]:
daily_df_enriched.write.mode("overwrite").parquet("../../Cleaned_data/daily_data.parquet")

# Beta Coefficiënt

Since our dataset contains multiple different Industries:
['Copper', 'Silver', 'Gold', 'Other Precious Metals & Mining', 'Other Industrial Metals & Mining']

It is necesarry to find a benchmark for each of these industries when computing the beta to increase accuracy.

In [76]:
print("Start date: ", start, "\nEnd date: ", end)

Start date:  2024-01-15 
End date:  2025-10-15


In [78]:
gold_benchmark = "GDX"
silver_benchmark = "SIL"
copper_benchmark = "COPX"
precious_metals_benchmark = "PICK"
industrial_mining_benchmark = "PICK"

benchmark_tickers = list(set([gold_benchmark, silver_benchmark, copper_benchmark, precious_metals_benchmark, industrial_mining_benchmark]))

In [79]:
# Create a benchmark map
benchmark_map = {
    "Gold": gold_benchmark,
    "Silver": silver_benchmark,
    "Copper": copper_benchmark,
    "Other Precious Metals & Mining": precious_metals_benchmark,
    "Other Industrial Metals & Mining": industrial_mining_benchmark,
}

In [80]:
benchmark_map_spark = F.create_map(
    *[F.lit(kv) for kv in sum(benchmark_map.items(), ())]
)

In [81]:
daily_df_enriched = daily_df_enriched.withColumn(
    "benchmark",
    benchmark_map_spark[F.col("industry")]
)
daily_df_enriched.show()

+------+---------------+----+-----+----+-----+--------+--------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+--------------------+--------+---------+
|ticker|Local Date (ET)|Open| High| Low|Close|  Volume|        daily_return|     intraday_return|      intraday_range|                 gap|up_day|down_day|volume_up|volume_down|days_since_start|biweekly_index|biweekly_period|            longName|industry|benchmark|
+------+---------------+----+-----+----+-----+--------+--------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+--------------------+--------+---------+
|ARG.TO|     2024-01-15|1.36| 1.38|1.35| 1.38| 71200.0|                NULL|0.014705882352941025| 0.02205882352941162|                NULL|     1|       0|  71200.0|       NULL|               0|        

Get data for all benchmarks

In [82]:
prices = yf.download(
    tickers=benchmark_tickers,
    start=start,
    end=end,
    auto_adjust=True
)[['Close', 'Open']].reset_index()

[*********************100%***********************]  4 of 4 completed


In [83]:
prices.columns = ['_'.join(col) for col in prices.columns]
prices.rename(columns={'Date_': 'Local Date (ET)'}, inplace=True)

prices.head()

,Local Date (ET),Close_COPX,Close_GDX,Close_PICK,Close_SIL,Open_COPX,Open_GDX,Open_PICK,Open_SIL
0,2024-01-16,34.942085,28.060568,37.966629,25.300415,35.606900,28.871052,38.386043,25.914881
1,2024-01-17,34.111061,27.190781,37.470955,24.481127,34.316372,27.625676,37.470955,24.959045
2,2024-01-18,34.120838,27.269852,37.585342,24.442112,33.993739,27.467532,37.442358,24.656689
3,2024-01-19,34.502132,27.368692,37.747383,24.461620,33.856867,27.487299,37.442354,24.627429
4,2024-01-22,33.700439,27.279736,37.251717,24.471373,34.003516,27.012868,37.270778,24.266549


In [86]:
prices_sdf = spark.createDataFrame(prices)

In [87]:
prices_sdf = add_biweekly_periods(prices_sdf)
prices_sdf = prices_sdf.drop('days_since_start', 'biweekly_index')
prices_sdf.show()

+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------+
|    Local Date (ET)|        Close_COPX|         Close_GDX|        Close_PICK|         Close_SIL|         Open_COPX|          Open_GDX|         Open_PICK|          Open_SIL|biweekly_period|
+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------+
|2024-01-16 00:00:00| 34.94208526611328| 28.06056785583496| 37.96662902832031|  25.3004150390625| 35.60690043777215|28.871052448052627|  38.3860431349048|  25.9148807257817|             P0|
|2024-01-17 00:00:00|34.111061096191406|27.190780639648438| 37.47095489501953| 24.48112678527832| 34.31637183640071|27.625675510726204| 37.47095489501953|24.959044966122537|             P0|
|2024-01-18 00:00:00|  34.1208381652832|27.2698516

## Merge on date and bi-weekly period

### Reshape Benchmark Table

In [93]:
benchmark_df = prices_sdf.select(
    "Local Date (ET)",
    "biweekly_period",
    F.expr("""
           stack(4, 
           'COPX', Open_COPX, Close_COPX,
           'GDX', Open_GDX, Close_GDX,
           'PICK', Open_PICK, Close_PICK,
           'SIL', Open_SIL, Close_SIL) as (benchmark, benchmark_open, benchmark_close)""")
).where(("benchmark_open IS NOT NULL") and ("benchmark_close IS NOT NULL"))

# Window per benchmark by date
w_bench = Window.partitionBy("benchmark").orderBy("Local Date (ET)")
# Compute Daily Return
benchmark_df = (
    benchmark_df  
    .withColumn(
        "benchmark_daily_return",
        (F.col("benchmark_close") / F.lag("benchmark_close").over(w_bench)) - 1
    )
)

benchmark_df.show()


+-------------------+---------------+---------+------------------+------------------+----------------------+
|    Local Date (ET)|biweekly_period|benchmark|    benchmark_open|   benchmark_close|benchmark_daily_return|
+-------------------+---------------+---------+------------------+------------------+----------------------+
|2024-01-16 00:00:00|             P0|     COPX| 35.60690043777215| 34.94208526611328|                  NULL|
|2024-01-17 00:00:00|             P0|     COPX| 34.31637183640071|34.111061096191406|  -0.02378290143799...|
|2024-01-18 00:00:00|             P0|     COPX| 33.99373944859161|  34.1208381652832|  2.866245955885915...|
|2024-01-19 00:00:00|             P0|     COPX| 33.85686742458294|34.502132415771484|   0.01117482075443954|
|2024-01-22 00:00:00|             P0|     COPX|34.003516145438276|   33.700439453125|  -0.02323604097815...|
|2024-01-23 00:00:00|             P0|     COPX|34.071951042575535| 34.47279739379883|  0.022918334395850337|
|2024-01-24 00:00:0

### Join with daily data

In [109]:
daily_df_enriched.show()

+------+---------------+----+-----+----+-----+--------+--------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+--------------------+--------+---------+
|ticker|Local Date (ET)|Open| High| Low|Close|  Volume|        daily_return|     intraday_return|      intraday_range|                 gap|up_day|down_day|volume_up|volume_down|days_since_start|biweekly_index|biweekly_period|            longName|industry|benchmark|
+------+---------------+----+-----+----+-----+--------+--------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+---------------+--------------------+--------+---------+
|ARG.TO|     2024-01-15|1.36| 1.38|1.35| 1.38| 71200.0|                NULL|0.014705882352941025| 0.02205882352941162|                NULL|     1|       0|  71200.0|       NULL|               0|        

In [94]:
joined_df = (
    daily_df_enriched
    .join(
        benchmark_df,
        on=[
            "Local Date (ET)",
            "benchmark",
            "biweekly_period"
        ],
        how="left"
    )
)

In [95]:
joined_df.show()

+---------------+---------+---------------+------+----+-----+----+-----+--------+--------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+--------------------+--------+------------------+------------------+----------------------+
|Local Date (ET)|benchmark|biweekly_period|ticker|Open| High| Low|Close|  Volume|        daily_return|     intraday_return|      intraday_range|                 gap|up_day|down_day|volume_up|volume_down|days_since_start|biweekly_index|            longName|industry|    benchmark_open|   benchmark_close|benchmark_daily_return|
+---------------+---------+---------------+------+----+-----+----+-----+--------+--------------------+--------------------+--------------------+--------------------+------+--------+---------+-----------+----------------+--------------+--------------------+--------+------------------+------------------+----------------------+
|     2024-01-18|  

In [111]:
final_df = aggregate_to_biweekly(joined_df)
final_df.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+
|ticker|biweekly_index|biweekly_period|            longname|industry|benchmark|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|benchmark_period_return|                beta|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|total_volume|  avg_daily_volume|max_daily_volume| volume_volatility|avg_volume_up_days|avg_volume_down_days|gap_count|        avg_gap_size|             max_gap|up_days|down_days|
+-

In [ ]:
final_df

root
 |-- ticker: string (nullable = true)
 |-- biweekly_period: string (nullable = true)
 |-- period_start: date (nullable = true)
 |-- period_end: date (nullable = true)
 |-- period_open: double (nullable = true)
 |-- period_close: double (nullable = true)
 |-- period_high: double (nullable = true)
 |-- period_low: double (nullable = true)
 |-- avg_close: double (nullable = true)
 |-- period_return: double (nullable = true)
 |-- benchmark_period_return: double (nullable = true)
 |-- beta: double (nullable = true)
 |-- price_volatility: double (nullable = true)
 |-- return_volatility: double (nullable = true)
 |-- avg_intraday_range: double (nullable = true)
 |-- max_intraday_range: double (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- avg_daily_volume: double (nullable = true)
 |-- max_daily_volume: double (nullable = true)
 |-- volume_volatility: double (nullable = true)
 |-- avg_volume_up_days: double (nullable = true)
 |-- avg_volume_down_days: double (nullable

Most people don't believe that stocks really have negative betas. But put options (insurances) or gold tend to have negative betas. A negative beta really only means negative covariance between the returns of your asset and the market, that is as the market goes up, your asset (gold or put option) drops in value and vice versa. You'd include an asset with low (or negative) market beta in your portfolio to hedge (offset the (systematic) risk from high beta positions) or to gamble on a declicing market.
https://quant.stackexchange.com/questions/57990/how-can-beta-be-negative

In [34]:
basetable = spark.read.parquet("../../Cleaned_data/biweekly_basetable.parquet")

In [31]:
biweekly_change.show()

+------+--------------+---------------+--------------------+
|ticker|biweekly_index|biweekly_period|            avg_%Chg|
+------+--------------+---------------+--------------------+
|ARG.TO|             0|             P0|  -0.305810109744827|
|ARG.TO|             1|             P1| -0.5971767724768162|
|ARG.TO|             2|             P2|  0.2758950896125816|
|ARG.TO|             3|             P3| 0.08927499458281644|
|ARG.TO|             4|             P4|  1.2325078746517593|
|ARG.TO|             5|             P5|  0.6520675714893737|
|ARG.TO|             6|             P6|  1.2934333128453068|
|ARG.TO|             7|             P7|0.025956881642292772|
|ARG.TO|             8|             P8|  0.7381104787800041|
|ARG.TO|             9|             P9| -0.6851516892693033|
|ARG.TO|            10|            P10| -1.1020973375798648|
|ARG.TO|            11|            P11|  -0.093496082472726|
|ARG.TO|            12|            P12|  0.9282145437944267|
|ARG.TO|            13| 

In [35]:
basetable.show()

+-------+--------------+---------------+--------------------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+-------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+---------+--------------------+
| ticker|biweekly_index|biweekly_period|            longname|            industry|benchmark|period_start|period_end|period_open|period_close|period_high|period_low|          avg_close|       period_return|benchmark_period_return|                beta|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|total_volume|  avg_daily_volume|max_daily_volum

In [32]:
joined = (
    basetable
    .join(biweekly_change, on=["ticker", "biweekly_index", "biweekly_period"], how="left")
)
joined.show()

+------+--------------+---------------+--------------------+--------+---------+------------+----------+-----------+------------+-----------+----------+------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+
|ticker|biweekly_index|biweekly_period|            longname|industry|benchmark|period_start|period_end|period_open|period_close|period_high|period_low|         avg_close|       period_return|benchmark_period_return|                beta|    price_volatility|   return_volatility|  avg_intraday_range|  max_intraday_range|total_volume|  avg_daily_volume|max_daily_volume| volume_volatility|avg_volume_u

In [33]:
joined.write.mode("overwrite").parquet("../../Cleaned_data/biweekly_basetable.parquet")